# 03: Probing Experiments — Within-Relation Truth Probe

**Task**: for each spatial relation with at least 50 training examples, restrict to the examples of that relation and train a logistic regression probe on frozen embeddings to predict VSR's True/False caption label.

This notebook:
1. Loads cached embeddings for the four conditions (SBERT, CLIP Text, CLIP Image, CLIP Concat)
2. Runs `probe_truth_by_relation` under two cross-validation schemes
3. Scores two dummy baselines (`most_frequent`, `stratified`) in the same folds
4. Saves per-model, comparison, summary and wins/ties/losses tables to `results/truth_probe/`

Settings: `LogisticRegression(C=1.0, max_iter=1000, solver="lbfgs", random_state=42)`, 5 folds, `random_state=42`, `min_samples=50`.

Cross-validation schemes:
- **stratified**: `StratifiedKFold(5, shuffle=True, random_state=42)`.
- **grouped**: `StratifiedGroupKFold(5, shuffle=True, random_state=42)`. Groups are captions for SBERT and CLIP Text, image files for CLIP Image, and joint caption-or-image connected components for CLIP Concat.

**Correction note**: an earlier version of this notebook fit a one-vs-rest relation-identity probe on all examples (`probe_relation_identity`, formerly `probe_by_relation_type`). Those results are kept in `results/relation_identity/`.

## 1. Setup

In [1]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
from dotenv import load_dotenv
import os

# Notebooks run from notebooks/ — resolve paths relative to project root
PROJECT_ROOT = (Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()).resolve()
sys.path.insert(0, str(PROJECT_ROOT))
load_dotenv(PROJECT_ROOT / ".env")

CACHE_DIR = Path(os.getenv("CACHE_DIR", "results/embeddings"))
if not CACHE_DIR.is_absolute():
    CACHE_DIR = PROJECT_ROOT / CACHE_DIR
OUT_DIR = PROJECT_ROOT / "results" / "truth_probe"
OUT_DIR.mkdir(parents=True, exist_ok=True)

np.random.seed(42)
pd.set_option("display.width", 250)
pd.set_option("display.max_rows", 100)

print(f"Project root: {PROJECT_ROOT}")
print(f"Cache directory: {CACHE_DIR}")
print(f"Output directory: {OUT_DIR}")

Project root: /Users/norair/Documents/Documents - Nora’s MacBook Air (2)/Code/Projects/spatial_probing
Cache directory: /Users/norair/Documents/Documents - Nora’s MacBook Air (2)/Code/Projects/spatial_probing/results/embeddings
Output directory: /Users/norair/Documents/Documents - Nora’s MacBook Air (2)/Code/Projects/spatial_probing/results/truth_probe


## 2. Load Embeddings and Dataset Labels

In [2]:
from datasets import load_dataset

MODELS = ["sbert", "clip_text", "clip_image", "clip_concat"]
DISPLAY = {"sbert": "SBERT", "clip_text": "CLIP Text", "clip_image": "CLIP Image", "clip_concat": "CLIP Concat"}

embeddings = {m: np.load(CACHE_DIR / f"{m}_vsr_train.npy") for m in MODELS}
for m, e in embeddings.items():
    print(f"  {m:12s} {e.shape}")

train_data = load_dataset("cambridgeltl/vsr_random")["train"]
relation_types = np.array(train_data["relation"])
binary_labels = np.array([int(v) for v in train_data["label"]])
captions = np.array(train_data["caption"])
image_files = np.array(train_data["image"])

for e in embeddings.values():
    assert e.shape[0] == len(train_data)

print(f"\nDataset: {len(train_data)} examples")
print(f"Relation types: {len(np.unique(relation_types))} unique")
print(f"Labels: {binary_labels.sum()} True, {len(binary_labels) - binary_labels.sum()} False")
print(f"Unique captions: {len(np.unique(captions))}, unique images: {len(np.unique(image_files))}")

  sbert        (7680, 768)
  clip_text    (7680, 512)
  clip_image   (7680, 512)
  clip_concat  (7680, 1024)



Dataset: 7680 examples
Relation types: 64 unique
Labels: 3876 True, 3804 False
Unique captions: 3762, unique images: 4968


## 3. Train Probes

Each condition is run twice: once with plain stratified folds and once with grouped folds.

In [3]:
from src.probing import probe_truth_by_relation, model_comparison_table

# Grouping key(s) per condition for the grouped run
GROUPS = {
    "sbert": captions,
    "clip_text": captions,
    "clip_image": image_files,
    "clip_concat": [captions, image_files],   # joint: shared caption OR shared image
}
GROUP_DESC = {"sbert": "caption", "clip_text": "caption", "clip_image": "image", "clip_concat": "caption+image (joint)"}

results = {}          # stratified folds
results_grouped = {}  # grouped folds
for m in MODELS:
    results[m] = probe_truth_by_relation(
        embeddings[m], relation_types, binary_labels, n_splits=5, C=1.0, min_samples=50
    )
    results_grouped[m] = probe_truth_by_relation(
        embeddings[m], relation_types, binary_labels, n_splits=5, C=1.0, min_samples=50,
        groups=GROUPS[m],
    )
    print(f"{m:12s} relations: {len(results[m])}  "
          f"mean macro F1 stratified: {results[m]['f1_macro_mean'].mean():.4f}  "
          f"grouped by {GROUP_DESC[m]}: {results_grouped[m]['f1_macro_mean'].mean():.4f}")

sbert        relations: 36  mean macro F1 stratified: 0.3696  grouped by caption: 0.4635


clip_text    relations: 36  mean macro F1 stratified: 0.3703  grouped by caption: 0.4449


clip_image   relations: 36  mean macro F1 stratified: 0.5057  grouped by image: 0.5026


clip_concat  relations: 36  mean macro F1 stratified: 0.4545  grouped by caption+image (joint): 0.4876


## 4. Group Sizes per Relation (grouped folds)

Number of groups and largest group per relation, for each grouping.

In [4]:
gs = results_grouped["sbert"][["relation", "n_samples"]].copy()
for m, name in [("sbert", "caption"), ("clip_image", "image"), ("clip_concat", "joint")]:
    g = results_grouped[m].set_index("relation")
    gs[f"{name}_groups"] = gs["relation"].map(g["n_groups"])
    gs[f"{name}_max"] = gs["relation"].map(g["max_group_size"])
    gs[f"{name}_max_pct"] = (100 * gs[f"{name}_max"] / gs["n_samples"]).round(1)
gs.to_csv(OUT_DIR / "group_sizes.csv", index=False)
print(gs.to_string(index=False))
print("\nLargest group as % of relation:",
      {n: float(gs[f"{n}_max_pct"].max()) for n in ["caption", "image", "joint"]})

            relation  n_samples  caption_groups  caption_max  caption_max_pct  image_groups  image_max  image_max_pct  joint_groups  joint_max  joint_max_pct
            touching        884             268           22              2.5           806          3            0.3           201         60            6.8
         in front of        528             271            9              1.7           515          2            0.4           258         22            4.2
              behind        498             251           12              2.4           487          2            0.4           240         15            3.0
               under        415             151           15              3.6           406          2            0.5           143         24            5.8
                  on        408             159           13              3.2           398          2            0.5           149         17            4.2
           on top of        349             138     

## 5. Comparison Table

One macro-F1 column per condition (stratified folds), the two dummy baselines scored in the same stratified folds, and one `_f1_grouped` column per condition. Sorted by `n_samples`.

In [5]:
comparison = model_comparison_table(results)
grouped_cmp = model_comparison_table(results_grouped)
for m in MODELS:
    comparison[f"{m}_f1_grouped"] = comparison["relation"].map(
        grouped_cmp.set_index("relation")[f"{m}_f1"]
    )

comparison.to_csv(OUT_DIR / "comparison.csv", index=False)
print(comparison.round(4).to_string(index=False))

            relation  n_samples  sbert_f1  clip_text_f1  clip_image_f1  clip_concat_f1  baseline_most_frequent_f1  baseline_stratified_f1  sbert_f1_grouped  clip_text_f1_grouped  clip_image_f1_grouped  clip_concat_f1_grouped
            touching        884    0.4044        0.4136         0.6464          0.6228                     0.3428                  0.5064            0.5099                0.5103                 0.6619                  0.6446
         in front of        528    0.4571        0.4765         0.5218          0.5142                     0.3342                  0.4976            0.5369                0.5309                 0.4965                  0.5566
              behind        498    0.3948        0.4035         0.5777          0.5378                     0.3465                  0.5357            0.4728                0.4558                 0.5923                  0.5634
               under        415    0.4272        0.4479         0.5259          0.4743              

## 6. Summary Statistics

`std across relations` is the standard deviation of the per-relation mean macro F1 over the 36 relations. `mean fold std` is the average, over relations, of the standard deviation across the 5 folds.

In [6]:
def summarize(label, cv, f1, fold_std=None):
    return {
        "condition": label,
        "cv": cv,
        "mean_f1": f1.mean(),
        "std across relations": f1.std(ddof=0),
        "median_f1": f1.median(),
        "min_f1": f1.min(),
        "max_f1": f1.max(),
        "mean fold std": np.nan if fold_std is None else fold_std.mean(),
        "n_relations": len(f1),
    }

rows = []
for m in MODELS:
    rows.append(summarize(DISPLAY[m], "stratified", results[m]["f1_macro_mean"], results[m]["f1_macro_std"]))
ref = results["sbert"]  # baselines depend only on labels and folds, identical across conditions
rows.append(summarize("Baseline: most_frequent", "stratified", ref["baseline_most_frequent_f1_macro"]))
rows.append(summarize("Baseline: stratified", "stratified", ref["baseline_stratified_f1_macro"]))
for m in MODELS:
    r = results_grouped[m]
    rows.append(summarize(DISPLAY[m], f"grouped ({GROUP_DESC[m]})", r["f1_macro_mean"], r["f1_macro_std"]))
    rows.append(summarize(f"Baseline: most_frequent [{DISPLAY[m]} folds]", f"grouped ({GROUP_DESC[m]})",
                          r["baseline_most_frequent_f1_macro"]))
    rows.append(summarize(f"Baseline: stratified [{DISPLAY[m]} folds]", f"grouped ({GROUP_DESC[m]})",
                          r["baseline_stratified_f1_macro"]))

summary = pd.DataFrame(rows)
summary.to_csv(OUT_DIR / "summary_statistics.csv", index=False)
print(summary.round(4).to_string(index=False))

                                  condition                              cv  mean_f1  std across relations  median_f1  min_f1  max_f1  mean fold std  n_relations
                                      SBERT                      stratified   0.3696                0.0584     0.3843  0.2511  0.4735         0.0528           36
                                  CLIP Text                      stratified   0.3703                0.0609     0.3683  0.2331  0.5023         0.0476           36
                                 CLIP Image                      stratified   0.5057                0.0881     0.5210  0.3608  0.6873         0.0690           36
                                CLIP Concat                      stratified   0.4545                0.0971     0.4219  0.3149  0.6885         0.0527           36
                    Baseline: most_frequent                      stratified   0.3499                0.0149     0.3467  0.3342  0.4121            NaN           36
                       Basel

## 7. Wins / Ties / Losses

Per-relation macro F1 compared for each pair of conditions. A tie is `abs(a - b) < 1e-9`. "wins" counts relations where `a` is higher.

In [7]:
from itertools import combinations

TOL = 1e-9

def wins_ties_losses(a: pd.Series, b: pd.Series) -> tuple[int, int, int]:
    diff = (a - b).to_numpy()
    ties = int((np.abs(diff) < TOL).sum())
    wins = int((diff >= TOL).sum())
    losses = int((diff <= -TOL).sum())
    return wins, ties, losses

wtl_rows = []
for cv_name, res in [("stratified", results), ("grouped", results_grouped)]:
    f1 = {m: res[m].set_index("relation")["f1_macro_mean"] for m in MODELS}
    for a, b in combinations(MODELS, 2):
        fb = f1[b].reindex(f1[a].index)
        w, t, l = wins_ties_losses(f1[a], fb)
        wtl_rows.append({"cv": cv_name, "a": a, "b": b, "a_wins": w, "ties": t, "a_losses": l,
                         "mean_diff_a_minus_b": (f1[a] - fb).mean()})
    for m in MODELS:
        for strategy in ["most_frequent", "stratified"]:
            base = res[m].set_index("relation")[f"baseline_{strategy}_f1_macro"]
            w, t, l = wins_ties_losses(f1[m], base.reindex(f1[m].index))
            wtl_rows.append({"cv": cv_name, "a": m, "b": f"baseline_{strategy}", "a_wins": w, "ties": t,
                             "a_losses": l, "mean_diff_a_minus_b": (f1[m] - base).mean()})

wtl = pd.DataFrame(wtl_rows)
wtl.to_csv(OUT_DIR / "wins_ties_losses.csv", index=False)
print(wtl.round(4).to_string(index=False))

        cv           a                      b  a_wins  ties  a_losses  mean_diff_a_minus_b
stratified       sbert              clip_text      15     1        20              -0.0008
stratified       sbert             clip_image       2     1        33              -0.1362
stratified       sbert            clip_concat       6     1        29              -0.0850
stratified   clip_text             clip_image       1     2        33              -0.1354
stratified   clip_text            clip_concat       3     2        31              -0.0842
stratified  clip_image            clip_concat      32     3         1               0.0512
stratified       sbert baseline_most_frequent      23     1        12               0.0197
stratified       sbert    baseline_stratified       2     0        34              -0.1248
stratified   clip_text baseline_most_frequent      19     2        15               0.0205
stratified   clip_text    baseline_stratified       2     0        34              -0.1240

## 8. Save Detailed Results

Per-relation results for each condition, including fold std (`f1_macro_std`, `accuracy_std`), precision, recall and both baselines.

In [8]:
for m in MODELS:
    results[m].to_csv(OUT_DIR / f"{m}_detailed.csv", index=False)
    results_grouped[m].to_csv(OUT_DIR / f"{m}_grouped_detailed.csv", index=False)
    print(f"Saved: {m}_detailed.csv, {m}_grouped_detailed.csv")

print("\nFiles in results/truth_probe/:")
for f in sorted(OUT_DIR.glob("*.csv")):
    print(f"  {f.name}")

Saved: sbert_detailed.csv, sbert_grouped_detailed.csv
Saved: clip_text_detailed.csv, clip_text_grouped_detailed.csv
Saved: clip_image_detailed.csv, clip_image_grouped_detailed.csv
Saved: clip_concat_detailed.csv, clip_concat_grouped_detailed.csv

Files in results/truth_probe/:
  clip_concat_detailed.csv
  clip_concat_grouped_detailed.csv
  clip_image_detailed.csv
  clip_image_grouped_detailed.csv
  clip_text_detailed.csv
  clip_text_grouped_detailed.csv
  comparison.csv
  group_sizes.csv
  sbert_detailed.csv
  sbert_grouped_detailed.csv
  summary_statistics.csv
  wins_ties_losses.csv
